In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/case-study/processed'

# load IBI data
baseline_ibi = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

# clean IBI data
def clean_ibi_data(ibi_data):
    ibi_data_clean = ibi_data[ibi_data['ibi'] > 0]
    return ibi_data_clean

baseline_ibi_clean = clean_ibi_data(baseline_ibi)
ibi_01_clean = clean_ibi_data(ibi_01)
ibi_02_clean = clean_ibi_data(ibi_02)
ibi_03_clean = clean_ibi_data(ibi_03)

# calculate RMSSD
def calculate_rmssd(ibi_values):
    diff_nn_intervals = np.diff(ibi_values)
    squared_diffs = diff_nn_intervals ** 2
    rmssd = np.sqrt(np.mean(squared_diffs))
    return rmssd

# calculate SDNN
def calculate_sdnn(ibi_values):
    sdnn = np.std(ibi_values, ddof=1)
    return sdnn

# compute HRV metrics
rmssd_baseline = calculate_rmssd(baseline_ibi_clean['ibi'].dropna().values)
sdnn_baseline = calculate_sdnn(baseline_ibi_clean['ibi'].dropna().values)

rmssd_01 = calculate_rmssd(ibi_01_clean['ibi'].dropna().values)
sdnn_01 = calculate_sdnn(ibi_01_clean['ibi'].dropna().values)

rmssd_02 = calculate_rmssd(ibi_02_clean['ibi'].dropna().values)
sdnn_02 = calculate_sdnn(ibi_02_clean['ibi'].dropna().values)

rmssd_03 = calculate_rmssd(ibi_03_clean['ibi'].dropna().values)
sdnn_03 = calculate_sdnn(ibi_03_clean['ibi'].dropna().values)

# display HRV metrics
print(f'Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms')
print(f'Session 1 RMSSD: {rmssd_01:.2f} ms, SDNN: {sdnn_01:.2f} ms')
print(f'Session 2 RMSSD: {rmssd_02:.2f} ms, SDNN: {sdnn_02:.2f} ms')
print(f'Session 3 RMSSD: {rmssd_03:.2f} ms, SDNN: {sdnn_03:.2f} ms')

# anxiety thresholds
rmssd_threshold = 20
sdnn_threshold = 50

# check HRV anxiety
anxiety_rmssd_baseline = rmssd_baseline < rmssd_threshold
anxiety_sdnn_baseline = sdnn_baseline < sdnn_threshold

anxiety_rmssd_01 = rmssd_01 < rmssd_threshold
anxiety_sdnn_01 = sdnn_01 < sdnn_threshold

anxiety_rmssd_02 = rmssd_02 < rmssd_threshold
anxiety_sdnn_02 = sdnn_02 < sdnn_threshold

anxiety_rmssd_03 = rmssd_03 < rmssd_threshold
anxiety_sdnn_03 = sdnn_03 < sdnn_threshold

# display anxiety results
print(f'Baseline RMSSD Anxiety: {"Yes" if anxiety_rmssd_baseline else "No"}, SDNN Anxiety: {"Yes" if anxiety_sdnn_baseline else "No"}')
print(f'Session 1 RMSSD Anxiety: {"Yes" if anxiety_rmssd_01 else "No"}, SDNN Anxiety: {"Yes" if anxiety_sdnn_01 else "No"}')
print(f'Session 2 RMSSD Anxiety: {"Yes" if anxiety_rmssd_02 else "No"}, SDNN Anxiety: {"Yes" if anxiety_sdnn_02 else "No"}')
print(f'Session 3 RMSSD Anxiety: {"Yes" if anxiety_rmssd_03 else "No"}, SDNN Anxiety: {"Yes" if anxiety_sdnn_03 else "No"}')

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

DATA = '../data/case-study/processed'

# load IBI data
ibi_baseline = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

def clean_ibi(df):
    ibi = df['ibi']
    # physiological bounds
    clean = ibi[(ibi > 300) & (ibi < 2000)]
    removed = len(ibi) - len(clean)
    return clean, removed

def hrv_metrics(ibi_values):
    nn = ibi_values.values
    sdnn = np.std(nn, ddof=1)
    diffs = np.diff(nn)
    rmssd = np.sqrt(np.mean(diffs**2))
    pnn50 = np.sum(np.abs(diffs) > 50) / len(diffs) * 100
    return sdnn, rmssd, pnn50

sessions = {
    'Baseline': ibi_baseline,
    'Session 01': ibi_01,
    'Session 02': ibi_02,
    'Session 03': ibi_03
}

# artifact rejection + HRV
print("=== IBI Artifact Rejection (300-2000ms bounds) ===\n")
results = {}
for name, df in sessions.items():
    clean, removed = clean_ibi(df)
    sdnn, rmssd, pnn50 = hrv_metrics(clean)
    results[name] = {'N': len(clean), 'Removed': removed, 'SDNN': sdnn, 'RMSSD': rmssd, 'pNN50': pnn50}
    print(f"{name}: {removed} artifacts removed, {len(clean)} valid beats")

# summary table
print("\n=== HRV Metrics (Time-Domain) ===\n")
hrv_table = pd.DataFrame(results).T.round(2)
print(hrv_table.to_string())

# compare sessions
print("\n=== Baseline vs Session Comparisons ===\n")
baseline_ibi, _ = clean_ibi(ibi_baseline)
for name in ['Session 01', 'Session 02', 'Session 03']:
    test_ibi, _ = clean_ibi(sessions[name])
    u_stat, p = stats.mannwhitneyu(baseline_ibi, test_ibi, alternative='two-sided')
    diff = test_ibi.mean() - baseline_ibi.mean()
    print(f"{name} vs Baseline: mean IBI diff={diff:.1f}ms, U p={p:.4f}")